# BRAMASTRA K8 — GPU session (T4 x2, execution only)

**GPU-only.** This notebook never generates data and never verifies on CPU time: attach the CPU-session outputs (K8 bundle dataset + build-report dataset) and the source dataset (or enable Internet for a pinned clone). One allocation, single run dir, E0 then full, summarize, export, package.

**Kaggle setup:** accelerator `GPU T4 x2`, Internet on if cloning. Budget: `--max-wall-minutes 600` (env `BRAMASTRA_MAX_WALL_MINUTES`), training stops 30 min before the wall for export. Precision is `fp32` by contract (T4 FP16 overflow history).

In [ ]:
import importlib.util
import json
import os
import re
import subprocess
import sys
import uuid
from pathlib import Path

WORKING = Path(os.environ.get('BRAMASTRA_WORKING', '/kaggle/working'))
INPUT = Path(os.environ.get('BRAMASTRA_INPUT', '/kaggle/input'))
GIT_URL = os.environ.get('BRAMASTRA_GIT_URL', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git')
GIT_REF = os.environ.get('BRAMASTRA_GIT_REF', 'BRAMASTRA')
AUTO_SOURCE_DIR = WORKING / 'bramastra-source'
MAX_WALL_MINUTES = int(os.environ.get('BRAMASTRA_MAX_WALL_MINUTES', '600'))

WORKING.mkdir(parents=True, exist_ok=True)


def _children(path):
    if not path.is_dir():
        return []
    try:
        return sorted(path.iterdir(), key=lambda item: item.name)
    except OSError:
        return []


def _is_source_tree(path):
    return (path / 'pyproject.toml').is_file() and (path / 'bramastra_lab').is_dir()


def _source_candidates():
    configured = os.environ.get('BRAMASTRA_REPO')
    if configured:
        yield Path(configured)
    yield Path.cwd()
    yield WORKING / 'An-Ra-the-new-AGI'
    for parent in (INPUT, WORKING):
        for child in _children(parent):
            yield child
            for grandchild in _children(child)[:50]:
                yield grandchild


REPO = next((path.resolve() for path in _source_candidates() if _is_source_tree(path)), None)
if REPO is None:
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._/-]{0,127}', GIT_REF):
        raise RuntimeError('BRAMASTRA_GIT_REF contains unsafe characters.')
    if AUTO_SOURCE_DIR.exists():
        raise RuntimeError(
            f'{AUTO_SOURCE_DIR} exists but is not a usable source tree. '
            'Attach the source dataset or start a fresh session.')
    print(f'cloning BRAMASTRA {GIT_REF!r} from {GIT_URL}...')
    clone = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', GIT_REF, GIT_URL, str(AUTO_SOURCE_DIR)],
        capture_output=True, text=True)
    if clone.returncode != 0 or not _is_source_tree(AUTO_SOURCE_DIR):
        detail = (clone.stderr or clone.stdout).strip()[-1200:]
        raise RuntimeError(
            'Automatic clone failed. Enable Internet in Kaggle Session options, '
            f'or attach the source dataset. Git detail: {detail}')
    REPO = AUTO_SOURCE_DIR.resolve()
    revision = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'],
                              capture_output=True, text=True, check=True).stdout.strip()
    print('cloned source revision:', revision)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if importlib.util.find_spec('bramastra_lab') is None:
    raise RuntimeError(f'Source at {REPO} cannot provide bramastra_lab.')

REQUIRED_MODULES = ('torch', 'numpy', 'pytest')
missing_modules = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing_modules:
    raise RuntimeError('Missing modules: ' + ', '.join(missing_modules) + '. Use a compatible Kaggle image.')

from bramastra_lab.research.campaigns.kaggle_env import (
    K8EnvironmentError,
    build_run_paths,
    check_git_ref,
    disk_contract,
    find_build_report,
    gpu_contract,
    package_artifacts,
    stream_command,
)

try:
    PATHS = build_run_paths()
    RUN_ID = PATHS.run_id
    INSTANCE_ID = PATHS.instance_id
    RUN_ROOT = PATHS.run_root
    RUN_DIR = PATHS.run_dir
    BUILD_REPORT_DIR = PATHS.build_report_dir
    EXPORT_DIR = PATHS.export_dir
    WORKING = PATHS.working
    INPUT = PATHS.input_root
except K8EnvironmentError as exc:
    raise RuntimeError(f'campaign environment refused: {exc}') from exc


def run_k8(label, *arguments):
    try:
        result = stream_command(
            label, [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8', *arguments],
            cwd=REPO,
            log_dir=RUN_ROOT,
            extra_env={'PYTHONPATH': str(REPO) + os.pathsep + os.environ.get('PYTHONPATH', '')},
        )
    except K8EnvironmentError as exc:
        raise RuntimeError(f'{label} could not start: {exc}') from exc
    if result.returncode != 0:
        try:
            _safety = subprocess.run(
                [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
                 'safety-snapshot', '--run-dir', str(RUN_ROOT / 'campaign' if 'RUN_DIR' not in dir() else RUN_DIR),
                 '--reason', f'failed-{label}'],
                cwd=REPO, capture_output=True, text=True, timeout=300)
            if _safety.stdout:
                print(_safety.stdout)
        except Exception as _safety_error:
            print(f'safety snapshot unavailable: {_safety_error}', file=sys.stderr)
        try:
            packed = package_artifacts(RUN_ROOT, WORKING, RUN_ID, INSTANCE_ID, f'failed-{label}')
            if packed is not None:
                print(f'failure artifacts packaged: {packed.archive}')
        except K8EnvironmentError as archive_error:
            print(f'could not package failure artifacts: {archive_error}', file=sys.stderr)
        raise RuntimeError(f'{label} failed with exit code {result.returncode} (full log: {result.log_path})')
    return result

check_git_ref(GIT_REF)

try:
    _gpu = gpu_contract(required=2)
    print('GPU count:', _gpu['count'])
    for _i, _name in enumerate(_gpu['names']):
        print(f'  cuda:{_i}:', _name)
    print('torch:', _gpu['torch'])
except K8EnvironmentError as exc:
    raise RuntimeError(str(exc)) from exc
try:
    _disk = disk_contract(WORKING)
    print(f"disk free under {WORKING}: {_disk['free_gib']} GiB")
except K8EnvironmentError as exc:
    print(f'disk check unavailable: {exc}')
import torch
from bramastra_lab.research.runtime.provenance import source_identity
print('source:', json.dumps(source_identity(), indent=2))
print('source root:', REPO)
print('run ID:', RUN_ID)
print('wall minutes:', MAX_WALL_MINUTES)
print('Phases: E0 E1 E2 E3 E4 E5 E6 (single allocation, export reserve 30)')
from bramastra_lab.research.config import tokenizer_identity
from bramastra_lab.research.campaigns.phases.ops import k8_campaign_config
print('config:', k8_campaign_config().identity())
print('tokenizer:', tokenizer_identity())
print('model: vocab260/L8/W256/H4/FFN704/ctx512 params6493952')


## 1. Attached K8 data bundle (no generation in this session)

Produced by the CPU session (`prepare` full counts + `validate`). This cell refuses — with an INPUT listing — when nothing valid is attached.

In [ ]:
from bramastra_lab.research.campaigns.kaggle_env import discover_bundle, is_k8_bundle
BUNDLE_PATH, _bundle_checked = discover_bundle()
if BUNDLE_PATH is None:
    print('No attached K8 bundle found. This GPU session never generates data.')
    print('Attach the bundle dataset produced by the CPU session, then rerun.')
    print('INPUT tree seen:')
    for _child in sorted(INPUT.iterdir(), key=lambda item: item.name) if INPUT.is_dir() else []:
        print('  input/', _child.name)
        if _child.is_dir():
            for _grand in sorted(_child.iterdir(), key=lambda item: item.name)[:20]:
                print('    ', _grand.name)
    raise RuntimeError('Missing bundle: attach the CPU-session K8 bundle dataset.')
BUNDLE_DIR = str(BUNDLE_PATH)
print('using K8 bundle:', BUNDLE_DIR)
try:
    _manifest = json.loads((BUNDLE_PATH / 'manifest.json').read_text(encoding='utf-8'))
    print('bundle schema:', _manifest.get('schema'), 'identity:', _manifest.get('identity', '<n/a>')[:16])
except OSError as exc:
    raise RuntimeError(f'cannot read bundle manifest: {exc}') from exc


## 2. Validate data

In [ ]:
run_k8('validate', 'validate', '--bundle', BUNDLE_DIR)


## 3. Build verification (reuse the CPU-session report when present)

Prefers the attached/imported `build_verification.json`; runs `verify-build --no-updates` only when no report exists. The campaign refuses to start without a verifying report.

In [ ]:
_report_file = BUILD_REPORT_DIR / 'build_verification.json'
_attached = sorted(INPUT.glob('*/build_verification.json')) + sorted(INPUT.glob('*/*/build_verification.json'))
if _report_file.is_file():
    print('reusing build report:', _report_file)
elif _attached:
    import shutil as _shutil
    BUILD_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    _shutil.copy2(_attached[0], _report_file)
    print('imported CPU-session build report:', _attached[0])
else:
    run_k8(
        'build verification', 'verify-build', '--data', BUNDLE_DIR,
        '--report-dir', str(BUILD_REPORT_DIR), '--no-updates',
        '--notebook', str(REPO / 'notebooks' / 'bramastra_k8_gpu.ipynb'))
print('build report:', _report_file)


## 4. E0 hardware-qualification gate (same allocation)

In [ ]:
run_k8(
    'E0 gate', 'run', '--mode', 'e0', '--run-dir', str(RUN_DIR),
    '--data', BUNDLE_DIR, '--max-wall-minutes', str(MAX_WALL_MINUTES),
    '--devices', 'cuda:0,cuda:1', '--build-report', str(find_build_report(RUN_ROOT, BUILD_REPORT_DIR)),
    '--precision', 'fp32')


## 5. Full campaign E0–E6 (same run dir, same allocation)

In [ ]:
run_k8(
    'full campaign', 'run', '--mode', 'full', '--run-dir', str(RUN_DIR),
    '--data', BUNDLE_DIR, '--max-wall-minutes', str(MAX_WALL_MINUTES),
    '--devices', 'cuda:0,cuda:1', '--build-report', str(find_build_report(RUN_ROOT, BUILD_REPORT_DIR)),
    '--precision', 'fp32')


## 6. Summarize

In [ ]:
run_k8('summarize', 'summarize', '--run-dir', str(RUN_DIR))


## 7. Export

In [ ]:
run_k8('export', 'export', '--run-dir', str(RUN_DIR), '--out', str(EXPORT_DIR))


## 8. X-factor probe pack (read-only architecture review)

Runs after export, before packaging: attention geometry, checkpoint lineage, allocation fidelity, phase accounting, tokenizer round trip. No allocation, zero optimizer updates, fresh report dir per run.

In [ ]:
import uuid as _uuid
_existing_xprobe = sorted(RUN_ROOT.glob('xprobe-*/xprobe_report.json'))
if _existing_xprobe:
    print('reusing xprobe report:', _existing_xprobe[-1])
    XPROBE_DIR = _existing_xprobe[-1].parent
else:
    XPROBE_DIR = RUN_ROOT / f'xprobe-{_uuid.uuid4().hex[:8]}'
    run_k8('xprobe', 'xprobe', '--run-dir', str(RUN_DIR), '--out', str(XPROBE_DIR), '--device', 'cuda:0')
_report_path = XPROBE_DIR / 'xprobe_report.json'
print('xprobe report:', _report_path)
_xr = json.loads(_report_path.read_text(encoding='utf-8'))
for _probe in _xr.get('probes', []):
    print(f"  {_probe['name']}: {_probe['status']}")
_p4 = next((x for x in _xr.get('probes', []) if x['name'] == 'P4-phase-accounting'), {})
if isinstance(_p4, dict) and 'total_device_minutes' in _p4:
    print(f"total device minutes: {_p4['total_device_minutes']}")
print('xprobe pass:', _xr.get('xprobe_pass'), 'failing:', _xr.get('failing'))


## 9. Package and download all artifacts

In [ ]:
from bramastra_lab.research.campaigns.kaggle_env import package_artifacts as _package
packed = _package(RUN_ROOT, WORKING, RUN_ID, INSTANCE_ID, 'completed')
if packed is None:
    raise RuntimeError('No K8 run artifacts exist yet. Run the campaign cells before packaging.')
ARCHIVE_PATH = packed.archive
print('Verified artifact archive:', ARCHIVE_PATH)
print('Archive receipt:', packed.receipt)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(ARCHIVE_PATH)))
except Exception as exc:
    print(f'Use Kaggle Output to download {ARCHIVE_PATH.name}: {exc}')
print('Save a Kaggle version, then download the ZIP from the Output panel if the link is not shown.')


## 10. Results ZIP (all experiment results, ~10MB, auto-download)

Packs ledger export, phase outputs, build/xprobe reports, manifests and logs — no checkpoint payload bytes, so the ZIP stays small enough for the Output panel. The file link downloads on click; Save a Version also preserves it in Output.


In [ ]:
RESULTS_ZIP = RUN_ROOT / f'{RUN_ID}-results.zip'
if RESULTS_ZIP.is_file():
    print('reusing results pack:', RESULTS_ZIP)
else:
    _pack_args = ['package-results', '--run-dir', str(RUN_DIR), '--out', str(RESULTS_ZIP), '--run-id', RUN_ID]
    for _extra in (BUILD_REPORT_DIR, EXPORT_DIR):
        _pack_args += ['--extra', str(_extra)]
    for _xp in sorted(RUN_ROOT.glob('xprobe-*')):
        _pack_args += ['--extra', str(_xp)]
    run_k8('package results', *_pack_args)
_receipt = json.loads((RESULTS_ZIP.with_suffix('.json')).read_text(encoding='utf-8'))
print(f"results pack: {RESULTS_ZIP.name} {_receipt['bytes'] // 1024} KiB, {_receipt['files']} files")
print('sha256:', _receipt['sha256'][:16] + '...')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULTS_ZIP)))
except Exception as exc:
    print(f'Use Kaggle Output to download {RESULTS_ZIP.name}: {exc}')


## 11. Safety sweep — last-window download (run anytime)

Snapshots completed work into a timestamped safety ZIP, rebuilds the small results pack over it, and exposes both for download. Run this cell at any time — especially before the session ends or after any failure — so completed experiments are never lost.

In [ ]:
import subprocess as _sp
_snap = _sp.run(
    [sys.executable, '-m', 'bramastra_lab.research.campaigns.k8',
     'safety-snapshot', '--run-dir', str(RUN_DIR), '--reason', 'notebook-safety-sweep'],
    cwd=REPO, capture_output=True, text=True, timeout=300)
if _snap.stdout:
    print(_snap.stdout)
RESULTS_ZIP = RUN_ROOT / f'{RUN_ID}-results.zip'
if RESULTS_ZIP.is_file():
    try:
        RESULTS_ZIP.unlink()
    except OSError:
        pass
_pack_args = ['package-results', '--run-dir', str(RUN_DIR), '--out', str(RESULTS_ZIP), '--run-id', RUN_ID]
for _extra in (BUILD_REPORT_DIR, EXPORT_DIR):
    _pack_args += ['--extra', str(_extra)]
for _xp in sorted(RUN_ROOT.glob('xprobe-*')):
    _pack_args += ['--extra', str(_xp)]
run_k8('package results (safety sweep)', *_pack_args)
print('SAFETY COMPLETE — download below, then Save a Version.')
try:
    from IPython.display import FileLink, display
    display(FileLink(str(RESULTS_ZIP)))
    _latest = RUN_DIR / 'safety' / 'safety-latest.json'
    if _latest.is_file():
        _ptr = json.loads(_latest.read_text(encoding='utf-8'))
        _szip = RUN_DIR / 'safety' / Path(_ptr.get('archive', '')).name
        if _szip.is_file():
            display(FileLink(str(_szip)))
except Exception as exc:
    print(f'Use Kaggle Output to download the ZIPs: {exc}')
